# 11장 성향점수 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 15장의 가상 코호트 10,000명으로 보정할 변수 고르기, 성향점수 추정과 겹침, 1:1 매칭과 가중, 표준화 평균차로 본 균형, 상대위험도·위험차·위험비와 강건 분산, 부트스트랩, 음성 대조 결과와 E-value를 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages(c("sandwich", "MatchIt"))를 실행합니다.


## a. 보정할 변수 고르기

### 셀 1. 보정할 변수를 늘려 가며 상대위험도 구하기 (15장 a 절)

In [ ]:
suppressPackageStartupMessages({library(sandwich); library(survival)})
d <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ps_cohort.csv")                       # 한 줄 = 환자 한 명
rr_ci <- function(fit, V) {                             # 상대위험도와 95% 신뢰구간
  b <- coef(fit)["drugA"]; se <- sqrt(V["drugA", "drugA"])
  round(exp(c(RR = unname(b), lower = unname(b) - 1.96 * se, upper = unname(b) + 1.96 * se)), 3)
}
sets <- list(none   = hosp_1y ~ drugA,
             agesex = hosp_1y ~ drugA + age + female,
             hf_ckd = hosp_1y ~ drugA + age + female + hf + ckd,
             all7   = hosp_1y ~ drugA + age + female + hf + ckd + prior_hosp + n_drug + tertiary,
             frail  = hosp_1y ~ drugA + age + female + hf + ckd + prior_hosp + n_drug + tertiary + frail)
t(sapply(sets, function(f) {
  fit <- glm(f, family = poisson, data = d)              # 수정 포아송 회귀
  rr_ci(fit, vcovHC(fit, type = "HC0"))                  # 강건 분산
}))

## b. 성향점수의 추정과 겹침

### 셀 2. 성향점수 모형과 두 군의 겹침 (15장 b 절)

In [ ]:
ps_fit <- glm(drugA ~ age + female + hf + ckd + prior_hosp + n_drug + tertiary,
              family = binomial, data = d)
summary(ps_fit)$coefficients
d$ps <- fitted(ps_fit)                                    # 환자마다의 성향점수
nobs(ps_fit)                                              # 분석에 들어간 인원
sapply(split(d$ps, d$drugA), range)                       # 군별 점수의 범위(0 = B, 1 = A)
r <- rank(d$ps); n1 <- sum(d$drugA); n0 <- sum(1 - d$drugA)
(sum(r[d$drugA == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)   # C 통계량
ex <- data.frame(age = c(78, 58), female = c(1, 0), hf = c(1, 0), ckd = c(1, 0),
                 prior_hosp = c(1, 0), n_drug = c(9, 4), tertiary = c(1, 0))
round(predict(ps_fit, newdata = ex, type = "response"), 4)   # 15장 b 절의 두 환자

## c. 매칭과 가중

### 셀 3. 로짓 성향점수로 1:1 매칭하기

In [ ]:
lg  <- qlogis(d$ps)                       # 성향점수의 로짓
cal <- 0.2 * sd(lg)                       # 허용 한계: 로짓 표준편차의 0.2배
ord <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch15_match_order.csv")$id   # A 사용자를 짝짓는 순서(15장과 같은 무작위 순서)
ctrl  <- which(d$drugA == 0)
avail <- rep(TRUE, length(ctrl))          # 아직 짝이 없는 B 사용자
pt <- pc <- integer(0)
for (t in match(ord, d$id)) {             # 비복원 1:1 최근접 매칭
  dist <- abs(lg[ctrl] - lg[t]); dist[!avail] <- Inf
  j <- which.min(dist)
  if (dist[j] <= cal) { pt <- c(pt, t); pc <- c(pc, ctrl[j]); avail[j] <- FALSE }
}
c(caliper = round(cal, 4), pairs = length(pt), unmatched_A = sum(d$drugA) - length(pt))
m <- rbind(cbind(d[pt, ], pair = seq_along(pt)), cbind(d[pc, ], pair = seq_along(pt)))   # 매칭 자료
un <- d$drugA == 1 & !(seq_len(nrow(d)) %in% pt)
round(rbind(unmatched = colMeans(d[un, c("ps", "age", "hf", "ckd")]),
            matched   = colMeans(d[pt, c("ps", "age", "hf", "ckd")])), 3)

### 셀 4. MatchIt으로 같은 설계 매칭하기

In [ ]:
suppressPackageStartupMessages(library(MatchIt))
set.seed(7)
mi <- matchit(drugA ~ age + female + hf + ckd + prior_hosp + n_drug + tertiary, data = d,
              method = "nearest", distance = "glm", link = "linear.logit",   # 로짓 성향점수
              caliper = 0.2, m.order = "random")                            # 표준편차의 0.2배
summary(mi)$nn
md <- match.data(mi)                     # subclass 열이 짝 번호
fit <- glm(hosp_1y ~ drugA, family = poisson, data = md)
rr_ci(fit, vcovCL(fit, cluster = ~ subclass, type = "HC0"))

### 셀 5. 역확률 가중치와 그 밖의 가중치

In [ ]:
pA <- mean(d$drugA)
d$w_ate <- ifelse(d$drugA == 1, 1 / d$ps, 1 / (1 - d$ps))               # 역확률 가중치
d$w_st  <- ifelse(d$drugA == 1, pA / d$ps, (1 - pA) / (1 - d$ps))       # 안정화 가중치
d$w_att <- ifelse(d$drugA == 1, 1, d$ps / (1 - d$ps))                   # A 사용자 기준(ATT)
d$w_ow  <- ifelse(d$drugA == 1, 1 - d$ps, d$ps)                         # 겹침 가중치
ess <- function(w) sum(w)^2 / sum(w^2)                                  # 유효 표본 크기
round(rbind(ate = range(d$w_ate), stabilized = range(d$w_st),
            att_B = range(d$w_att[d$drugA == 0])), 3)
round(c(sum_A = sum(d$w_ate[d$drugA == 1]), sum_B = sum(d$w_ate[d$drugA == 0]),
        ess_A = ess(d$w_ate[d$drugA == 1]), ess_B = ess(d$w_ate[d$drugA == 0])), 1)

## d. 균형 진단과 효과 추정

### 셀 6. 표준화 평균차로 균형 보기

In [ ]:
smd <- function(x, g, w = rep(1, length(x))) {        # 가중 평균과 가중 분산으로 구한 SMD
  m1 <- weighted.mean(x[g == 1], w[g == 1]); m0 <- weighted.mean(x[g == 0], w[g == 0])
  v1 <- weighted.mean((x[g == 1] - m1)^2, w[g == 1]); v0 <- weighted.mean((x[g == 0] - m0)^2, w[g == 0])
  (m1 - m0) / sqrt((v1 + v0) / 2)
}
vars <- c("age", "female", "hf", "ckd", "prior_hosp", "n_drug", "tertiary", "frail")
round(sapply(vars, function(v) c(before  = smd(d[[v]], d$drugA),
                                 matched = smd(m[[v]], m$drugA),
                                 iptw    = smd(d[[v]], d$drugA, d$w_ate))), 3)

### 셀 7. 매칭과 가중 자료의 상대위험도와 위험차

In [ ]:
fm <- glm(hosp_1y ~ drugA, family = poisson, data = m)
fw <- glm(hosp_1y ~ drugA, family = poisson, data = d, weights = w_ate)
ft <- glm(hosp_1y ~ drugA, family = poisson, data = d, weights = w_att)
fo <- glm(hosp_1y ~ drugA, family = poisson, data = d, weights = w_ow)
rbind(matched = rr_ci(fm, vcovCL(fm, cluster = ~ pair, type = "HC0")),   # 짝 번호로 묶은 강건 분산
      iptw    = rr_ci(fw, vcovHC(fw, type = "HC0")),
      att     = rr_ci(ft, vcovHC(ft, type = "HC0")),
      overlap = rr_ci(fo, vcovHC(fo, type = "HC0")),
      iptw_model_based = rr_ci(fw, vcov(fw)))                             # 가중치를 사람 수로 셈
gw <- glm(hosp_1y ~ drugA, family = gaussian, data = d, weights = w_ate)   # 위험차
round(coef(gw)["drugA"] + c(0, -1.96, 1.96) * sqrt(vcovHC(gw, type = "HC0")["drugA", "drugA"]), 4)

### 셀 8. 매칭과 가중 자료의 위험비

In [ ]:
hr <- function(fit) round(exp(c(HR = unname(coef(fit)["drugA"]), confint(fit)["drugA", ])), 3)
rbind(crude   = hr(coxph(Surv(hosp_day, hosp_1y) ~ drugA, data = d)),
      iptw    = hr(coxph(Surv(hosp_day, hosp_1y) ~ drugA, data = d, weights = w_ate)),  # 강건 분산이 기본
      matched = hr(coxph(Surv(hosp_day, hosp_1y) ~ drugA, data = m, cluster = pair)),
      strata  = hr(coxph(Surv(hosp_day, hosp_1y) ~ drugA + strata(pair), data = m)))    # 짝을 층으로

### 셀 9. 부트스트랩 신뢰구간

In [ ]:
set.seed(15)
boot_lrr <- replicate(500, {                       # 성향점수 추정까지 다시 하는 부트스트랩
  b <- d[sample(nrow(d), replace = TRUE), ]
  p <- fitted(glm(drugA ~ age + female + hf + ckd + prior_hosp + n_drug + tertiary,
                  family = binomial, data = b))
  w <- ifelse(b$drugA == 1, 1 / p, 1 / (1 - p))
  log(weighted.mean(b$hosp_1y[b$drugA == 1], w[b$drugA == 1]) /
      weighted.mean(b$hosp_1y[b$drugA == 0], w[b$drugA == 0]))
})
round(exp(quantile(boot_lrr, c(0.025, 0.975))), 3)
round(c(boot_se = sd(boot_lrr), robust_se = sqrt(vcovHC(fw, type = "HC0")["drugA", "drugA"])), 4)

## e. 남는 교란과 민감도 분석

### 셀 10. 음성 대조 결과로 남은 교란 찾기

In [ ]:
fnc <- glm(injury_ed ~ drugA, family = poisson, data = d)                     # 음성 대조 결과
fnm <- glm(injury_ed ~ drugA, family = poisson, data = m)
fnw <- glm(injury_ed ~ drugA, family = poisson, data = d, weights = w_ate)
rbind(crude   = rr_ci(fnc, vcovHC(fnc, type = "HC0")),
      matched = rr_ci(fnm, vcovCL(fnm, cluster = ~ pair, type = "HC0")),
      iptw    = rr_ci(fnw, vcovHC(fnw, type = "HC0")))

### 셀 11. E-value 구하기

In [ ]:
evalue <- function(rr) { rr <- ifelse(rr < 1, 1 / rr, rr); rr + sqrt(rr * (rr - 1)) }
ev_ci  <- function(lo, hi) if (lo <= 1 & hi >= 1) 1 else if (hi < 1) evalue(hi) else evalue(lo)   # 1에 가까운 끝
est <- function(fit, V) {                     # 반올림하지 않은 추정값과 신뢰한계
  b <- unname(coef(fit)["drugA"]); se <- sqrt(V["drugA", "drugA"])
  exp(c(b, b - 1.96 * se, b + 1.96 * se))
}
f0 <- glm(hosp_1y ~ drugA, family = poisson, data = d)
e  <- rbind(crude   = est(f0, vcovHC(f0, type = "HC0")),
            matched = est(fm, vcovCL(fm, cluster = ~ pair, type = "HC0")),
            iptw    = est(fw, vcovHC(fw, type = "HC0")))
round(cbind(E_point = evalue(e[, 1]), E_limit = apply(e, 1, function(x) ev_ci(x[2], x[3]))), 3)

## f. 과제 정답

### 셀 12. 과제 1 정답: 논문 결과의 E-value

In [ ]:
round(c(point = evalue(0.60), limit = evalue(0.64)), 2)   # HR 0.60 (95% CI 0.53–0.64)

### 셀 13. 과제 2 정답: 가중치 자르기

In [ ]:
q <- quantile(d$w_st, c(0.01, 0.99))
d$w_tr <- pmin(pmax(d$w_st, q[1]), q[2])             # 1·99번째 백분위수에서 자름
round(q, 3)
ftr <- glm(hosp_1y ~ drugA, family = poisson, data = d, weights = w_tr)
rr_ci(ftr, vcovHC(ftr, type = "HC0"))